## Load Packages, Silently

In [ ]:
## Create a vector of the required packages for this analysis
req_packages <- c("clusterProfiler", "cowplot", "ggplot2", "ggpubr", 
                  "gridExtra", "janitor", "RColorBrewer", "stringr", 
                  "tidyverse", "vegan")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Set up data, for all figures

In [ ]:
## load in parasperm content information and the obscura tree
parasperm <- read_csv("../workflow/tools/parasperm_proportion.csv", skip = 1) %>%
    janitor::clean_names() %>%
    mutate(species = str_replace(species, "D. ", "Drosophila_")) %>%
    rename(parasperm = 2) %>%
    select(species, parasperm)
tree <- ape::read.tree("../workflow/tools/obscura_tree.nwk")

In [ ]:
## load in results for evogenex and cagee and cluster information
credible_df <- read_csv("../workflow/CAGEE/credible_genes.csv")
evogenex_raw <- read_csv("../workflow/EvoGeneX/evogenex_results.csv") 
evogenex <- evogenex_raw %>%
    na.omit()
pic_clusters <- read_tsv("intermediate/pic_clusters.csv")

In [ ]:
## load in vep results
raw_variants <- read_tsv("variant_effect_output.txt", skip = 95) %>%
    janitor::clean_names()

## clean up the column information
all_variants <- raw_variants %>%
    separate_wider_delim(extra, ";", names = c("impact", "distance", "strand", "class", "symbol", "biotype"), 
                         too_few = "align_start", too_many = "drop") %>%
    mutate(biotype = case_when(grepl("BIOTYPE", symbol) ~ symbol,
                               TRUE ~ biotype),
           symbol = case_when(grepl("SYMBOL", class) ~ class,
                              TRUE ~ symbol),
           class = case_when(grepl("VARIANT_CLASS", distance) ~ distance,
                             grepl("VARIANT_CLASS", strand) ~ strand,
                             TRUE ~ class),
           strand = case_when(grepl("STRAND", distance) ~ distance,
                              TRUE ~ strand),
           distance = case_when(!grepl("DISTANCE", distance) ~ NA,
                                TRUE ~ distance)) %>%
    mutate(across(everything(), ~ str_replace(., ".*=", ""))) %>%
    separate_wider_delim(number_uploaded_variation, "_", names = c("chromosome", "position", "change"), too_few = "align_start", too_many = "drop") %>%
    mutate(position = as.integer(position))

## filter for only variants on the XR
variants <- all_variants %>%
    filter(grepl("ChrX", chromosome)) %>%
    filter(grepl("ChrX.1", chromosome) | grepl("ChrX.4", chromosome))

In [ ]:
## load in tau data
tau <- read.csv("../workflow/yang2018_rnaseq/results/tau_tissue_0.50.csv")
testes_genes <- tau %>%
    filter(tau_tissue == "male_gonad") %>%
    pull("gene")

In [ ]:
## load in LogFC data
comparison_norm <- read_csv("../workflow/multigenome_rnaseq/results/logfc_normalized.csv")

In [ ]:
## load in GO term assignment for differentially expressed genes in the mutant
pseudoobscura_mutant_go <- read_csv("intermediate/pseudoobscura_mutant_go.csv")

In [ ]:
## load in GO term information for EvoGeneX and CAGEE
credible_go <- read_csv("intermediate/credible_go.csv")
adaptive_go <- read_csv("intermediate/adaptive_go.csv")

In [ ]:
## load in GO terms for branch associated genes that are upregulated in the mutant
upcred_go <- read_csv("intermediate/credible_up_go.csv")

In [ ]:
## load in cluster quality information
k_df <- read_csv("intermediate/cluster_number_df.csv")

In [ ]:
## load RNA reads from across the obscura group
reads <- read_tsv("../workflow/tools/obscura_group_reads.tsv")

## Create Figures

### Figure 1

In [ ]:
## get the percent parasperm measured
parasperm_percent <- parasperm %>%
    rename(label = species)

## annotate tree with percent parasperm
tree_label <- tree %>%
    as_tibble() %>%
    left_join(parasperm_percent, by = "label") %>%
    mutate(label = str_replace_all(label, "_", " "))

figure1a <- ggtree(tree) %<+% 
    tree_label +
    xlim(NA, 20) +
    labs(title = "")

figure1b <- tree_label %>%
    filter(grepl("Drosophila", label)) %>%
    ggplot(aes(x = factor(label, levels = c("Drosophila pseudoobscura", "Drosophila azteca", "Drosophila affinis", "Drosophila bifasciata",
                                                                   "Drosophila obscura", "Drosophila guanche", "Drosophila subobscura", "Drosophila madeirensis")))) +
        geom_bar(aes(y = percent_parasperm), stat = "identity") +
        geom_text(aes(y = percent_parasperm + 2, label = percent_parasperm), size = 6) +
        labs(y = "Percent Parasperm") +
        theme_minimal() +
        theme(text = element_text(size = 20),
              axis.title.y = element_blank(),
              axis.text.y = element_text(face = "italic", hjust = 0)) +
        coord_flip()

figure1 <- plot_grid(figure1a, NULL, figure1b, rel_widths = c(1, -0.3, 1), nrow = 1)
figure1

In [ ]:
ggsave("figures/figure1a.png", figure1a, width = 10, height = 9)
ggsave("figures/figure1b.png", figure1b, width = 10, height = 10)

### Figure 2

In [ ]:
## create list of representative genes for adaptive
rep_genes <- c("LOC26532845", "LOC6897231", "LOC6903509")

## plot representative genes
neutral_gene <- pic_clusters %>%
    filter(gene == rep_genes[1]) %>%
    ggplot(aes(x = parasperm, y = scale_change)) +
    geom_line() +
    ylim(-1.5, 2.5) +
    labs(title = paste0(rep_genes[1], "\n(Neutral Evolution)"),
         x = "Percent Parasperm", y = "Scaled Expression") +
    theme_minimal() +
    theme(plot.title = element_text(size = 15, hjust = 0.5))
constrained_gene <- pic_clusters %>%
    filter(gene == rep_genes[2]) %>%
    ggplot(aes(x = parasperm, y = scale_change)) +
    geom_line() +
    ylim(-1.5, 2.5) +
    labs(title = paste0(rep_genes[2], "\n(Constrained Evolution)"),
         x = "Percent Parasperm", y = "Scaled Expression") +
    theme_minimal() +
    theme(plot.title = element_text(size = 15, hjust = 0.5))
adaptive_gene <- pic_clusters %>%
    filter(gene == rep_genes[3]) %>%
    ggplot(aes(x = parasperm, y = scale_change)) +
    geom_line() +
    ylim(-1.5, 2.5) +
    labs(title = paste0(rep_genes[3], "\n(Parasperm Adaptive Evolution)"),
         x = "Percent Parasperm", y = "Scaled Expression") +
    theme_minimal() +
    theme(plot.title = element_text(size = 15, hjust = 0.5))

## pull statistics from evogenex
neutral_constrained_q <- evogenex %>%
    filter(gene == rep_genes[1]) %>%
    pull("constraint_qvalue") %>%
    signif(digits = 2)
neutral_adaptive_q <- evogenex %>%
    filter(gene == rep_genes[1]) %>%
    pull("adaptive2_qvalue") %>%
    signif(digits = 2)

constrained_constrained_q <- evogenex %>%
    filter(gene == rep_genes[2]) %>%
    pull("constraint_qvalue") %>%
    signif(digits = 2)
constrained_adaptive_q <- evogenex %>%
    filter(gene == rep_genes[2]) %>%
    pull("adaptive2_qvalue") %>%
    signif(digits = 2)

adaptive_constrained_q <- evogenex %>%
    filter(gene == rep_genes[3]) %>%
    pull("constraint_qvalue") %>%
    signif(digits = 2)
adaptive_adaptive_q <- evogenex %>%
    filter(gene == rep_genes[3]) %>%
    pull("adaptive2_qvalue") %>%
    signif(digits = 2)


## annotate graphs with statistics
neutral_annotated <- neutral_gene +
    annotate("text", x = 32, y = 2.5, hjust = 0, size = 5,
             label = paste0("Constrained Q Value = ", neutral_constrained_q)) +
    annotate("text", x = 32, y = 2.25, hjust = 0, size = 5,
             label = paste0("Adaptive Q Value = ", neutral_adaptive_q))
constrained_annotated <- constrained_gene +
    annotate("text", x = 32, y = 2.5, hjust = 0, size = 5,
             label = paste0("Constrained Q Value = ", constrained_constrained_q)) +
    annotate("text", x = 32, y = 2.25, hjust = 0, size = 5,
             label = paste0("Adaptive Q Value = ", constrained_adaptive_q))
adaptive_annotated <- adaptive_gene +
    annotate("text", x = 32, y = 2.5, hjust = 0, size = 5,
             label = paste0("Constrained Q Value = ", adaptive_constrained_q)) +
    annotate("text", x = 32, y = 2.25, hjust = 0, size = 5,
             label = paste0("Adaptive Q Value = ", adaptive_adaptive_q))

figure2a <- neutral_annotated + constrained_annotated + adaptive_annotated
figure2a

In [ ]:
## get number of credible changes at each branch
branch_change <- credible_df %>%
    clean_names() %>%
    separate_wider_delim(number_taxon_id, "<", names = c("name", "node")) %>%
    select(-name) %>%
    mutate(node = str_replace(node, ">", ""),
           node = as.numeric(node),
           change = paste0("+", increase, "/-", decrease)) %>%
    select(c("node", "change"))

## compile metadata into tibble
tree_label <- tree %>%
    as_tibble() %>%
    left_join(branch_change, by = "node") %>%
    mutate(label = str_replace_all(label, "_", " ")) %>%
    add_column(color = c("low", "low", "medium", "medium", "medium", "high", "high", "medium",
                            "low", "low", "low", "medium", "medium", "medium", "high"))
tree_label[11, "change"] <- NA

## plot tree
figure2b <- ggtree(tree) %<+% 
    tree_label +
    geom_tiplab(geom = "text", aes(label = paste0("italic(\'", label, "\')")), parse = TRUE,
                hjust = -0.12, size = 8) +
    geom_label(aes(x = branch, label = change), label.size = 0.01) +
    geom_nodepoint(aes(color = color), size = 5) +
    geom_tippoint(aes(color = color), size = 5) +
    xlim(NA, 20) +
    labs(title = "B", color = "Parasperm Percentage Category") +
    scale_color_manual(values = c("red", "blue", "purple")) +
    theme(plot.title = element_text(size = 50),
          legend.position = "bottom",
          legend.title = element_text(size = 20),
          legend.text = element_text(size = 20))
figure2b

In [ ]:
ggsave("figures/figure2a.png", figure1a, width = 18, height = 10)
ggsave("figures/figure2b.png", figure2b, width = 18, height = 10)

### Figure 3

In [ ]:
## annotate mutation types
variant_type <- variants %>%
    separate_longer_delim(consequence, delim = ",") %>%
    mutate(type = case_when(grepl("UTR", consequence) ~ "UTR",
                            grepl("splice", consequence) ~ "splicing",
                            grepl("intron", consequence) ~ "intron",
                            grepl("frameshift", consequence) ~ "frameshift",
                            grepl("inframe", consequence) ~ "inframe",
                            grepl("start_lost", consequence) | grepl("stop_gained", consequence) ~ "nonsense",
                            grepl("missense", consequence) ~ "missense",
                            grepl("protein_altering", consequence) ~ "protein altering",
                            grepl("stream_gene_variant", consequence) ~ "distal"))
                        
## create graphs to look at classifications of data
variant_type_graph <- variants %>%
    separate_longer_delim(consequence, delim = ",") %>%
    group_by(consequence) %>%
    summarize(count = n()) %>%
    arrange(desc(count)) %>%
    head(10) %>%
    mutate(consequence = str_replace_all(consequence, "_", " "),
           consequence = str_wrap(consequence, width = 25)) %>%
    ggplot(aes(x = reorder(consequence, count), y = count)) +
        geom_bar(stat = "identity") +
        labs(x = "Variant Type", y = "Number of Genes") +
        theme_bw() +
        theme(panel.grid = element_blank()) +
        coord_flip()
mutation_type_graph <- variants %>%
    group_by(class) %>%
    summarize(count = n()) %>%
    mutate(class = str_replace_all(class, "_", " ")) %>%
    ggplot(aes(x = reorder(class, count), y = count)) +
        geom_bar(stat = "identity") +
        labs(x = "Mutation Type", y = "Number of Mutations") +
        theme_bw()+
        theme(panel.grid = element_blank()) +
        coord_flip()
variant_number_graph <- variant_type_number %>%
    ggplot(aes(x = type, y = n)) +
        geom_bar(stat = "identity") +
        labs(x = "Mutation Type", y = "Number of Genes") +
        theme_bw() +
        theme(panel.grid = element_blank())

## add figure annotations
figure3a <- variant_type_graph +
    labs(title = "A") +
    theme(text = element_text(size = 15),
          plot.title = element_text(size = 22))
figure3b <- mutation_type_graph +
    labs(title = "B") +
    theme(text = element_text(size = 15),
          plot.title = element_text(size = 22))

figure3c <- variant_number_graph +
    labs(title = "C") +
    theme(text = element_text(size = 15),
          plot.title = element_text(size = 22))

## plot graphs together
figure3 <- (figure3a + figure3b) / figure2c
figure3

In [ ]:
ggsave("figures/figure3.png", figure2, height = 10, width = 15)

### Figure 4

In [ ]:
## combine differential expression and parasperm data
annotated_logfc <- reads$gene_id %>%
    as.data.frame() %>%
    rename(gene = 1) %>%
    left_join(logfc, by = "gene") %>%
    mutate(logfc = -(logfc),
           credible = case_when(gene %in% credible_genes ~ "credible",
                                TRUE ~ "non-credible"),
           adaptive = case_when(gene %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive"),
           significance = case_when(is.na(significance) ~ "not significant",
                                    TRUE ~ significance),
           color = case_when(is.na(color) ~ "not significant",
                             TRUE ~ color),
           direction = case_when(significance == "significant" & logfc < 0 ~ "hypo",
                                 significance == "significant" & logfc > 0 ~ "hyper",
                                 TRUE ~ "not significant"),
           testes_specificity = case_when(gene %in% testes_genes ~ "testes",
                                          TRUE ~ "other"),
           x = case_when(gene %in% x_genes ~ "x",
                         TRUE ~ "other")) %>%
    unique()

In [ ]:
## save number of genes and labels
gene_numbers <- annotated_logfc %>%
    count(direction) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    as.list()
up_rawlabel <- paste0("Up in Mutant = ", gene_numbers["hyper"])
down_rawlabel <- paste0("Down in Mutant = ", gene_numbers["hypo"])
not_rawlabel <- paste0("Not significant = ", gene_numbers["not significant"])

## get an unannotated logfc
logfc_raw <- annotated_logfc %>%
    na.omit() %>%
    ggplot(aes(x = logfc, y = -log10(pvalue), color = direction)) +
        geom_point() +
        labs(x = "LogFC", y = "-log10(P Value)") +
        xlim(-15, 15) +
        ylim(0, 15) +
        scale_color_manual(values = c("steelblue2", "steelblue4", "light gray")) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none")
figure4a <- logfc_raw +
    annotate("text", x = -15, y = 14, hjust = 0,
             size = 10, label = "A") +
    annotate("text", x = -15, y = 12, hjust = 0,
             size = 2, label = up_rawlabel) +
    annotate("text", x = -15, y = 11, hjust = 0,
             size = 2, label = down_rawlabel)
    # annotate("text", x = -15, y = 12, hjust = 0,
    #          label = not_rawlabel)

In [ ]:
## now look at testes, separating increase and decrease
testes_up <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(testes_specificity, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("testes_specificity") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
testes_down <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(testes_specificity, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("testes_specificity") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()

## now look at x, separating increase and decrease
x_up <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(x, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("x") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
x_down <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(x, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("x") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()

In [ ]:
## sub sample logfc based on characteristics of genes
logfc_testes <- annotated_logfc %>%
    filter(testes_specificity == "testes") %>%
    na.omit %>%
    ggplot(aes(x = logfc, y = -log10(pvalue), color = direction)) +
        geom_point() +
        labs(title = "Testes Specific", x = "LogFC", y = "-log10(P Value)") +
        xlim(-15, 15) +
        ylim(0, 15) +
        scale_color_manual(values = c("steelblue2", "steelblue4", "light gray")) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              plot.title = element_text(hjust = 0.5, size = 18))
logfc_x <- annotated_logfc %>%
    filter(x == "x") %>%
    na.omit %>%
    ggplot(aes(x = logfc, y = -log10(pvalue), color = direction)) +
        geom_point() +
        labs(title = "On X Chromosome", x = "LogFC", y = "-log10(P Value)") +
        xlim(-15, 15) +
        ylim(0, 15) +
        scale_color_manual(values = c("steelblue2", "steelblue4", "light gray")) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              plot.title = element_text(hjust = 0.5, size = 18))

## add graph details to testes specific
testes_numbers <- annotated_logfc %>%
    filter(testes_specificity == "testes") %>%
    count(direction) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    as.list()
up_testeslabel <- paste0("Up in Mutant = ", testes_numbers["hyper"], "/", length(testes_genes))
up_testesp <- paste0("p = ", signif(testes_up$p.value, digits = 4))
down_testeslabel <- paste0("Down in Mutant = ", testes_numbers["hypo"], "/", length(testes_genes))
down_testesp <- paste0("p = ", signif(testes_down$p.value, digits = 4))

figure4b <- logfc_testes +
    annotate("text", x = -15, y = 14, hjust = 0,
             size = 10, label = "B") +
    annotate("text", x = -15, y = 12, hjust = 0,
             size = 2, label = up_testeslabel) +
    annotate("text", x = -14, y = 11.5, hjust = 0,
             size = 2, label = up_testesp) +
    annotate("text", x = -15, y = 10, hjust = 0,
             size = 2, label = down_testeslabel) +
    annotate("text", x = -14, y = 9.5, hjust = 0,
             size = 2, label = down_testesp)

## add graph details to x
x_numbers <- annotated_logfc %>%
    filter(x == "x") %>%
    count(direction) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    as.list()
up_xlabel <- paste0("Up in Mutant = ", x_numbers["hyper"], "/", length(x_genes))
up_xp <- paste0("p = ", signif(x_up$p.value, digits = 4))
down_xlabel <- paste0("Down in Mutant = ", x_numbers["hypo"], "/", length(x_genes))
down_xp <- paste0("p = ", signif(x_down$p.value, digits = 4))

figure4c <- logfc_x +
    annotate("text", x = -15, y = 14, hjust = 0,
             size = 10 , label = "C") +
    annotate("text", x = -15, y = 12, hjust = 0,
             size = 2, label = up_xlabel) +
    annotate("text", x = -14, y = 11.5, hjust = 0,
             size = 2, label = up_xp) +
    annotate("text", x = -15, y = 10, hjust = 0,
             size = 2, label = down_xlabel) +
    annotate("text", x = -14, y = 9.5, hjust = 0,
             size = 2, label = down_xp)

In [ ]:
## plot them together
layout <- "#AAAA#
           #AAAA#
           #AAAA#
           BBBCCC
           BBBCCC"
figure4 <- logfc_rawannotated + logfc_testesannotated + logfc_xannotated +
    plot_layout(design = layout)
figure4

In [ ]:
ggsave("figures/figure4.png", figure4, height = 10, width = 8)

### Figure 5

In [ ]:
## create list of cellular compartments
compartments <- c("mitochondrial inner membrane", "extracellular region", "mitochondrion", "plasma membrane", "ribosome", 
                 "extracellular space", "organelle inner membrane", "biological_process", "cellular_component", "ribosomal subunit", 
                 "molecular_function", "cytosol", "intracellular organelle")

## create dot plots
pseudoobscura_mutant_up <- pseudoobscura_mutant_go %>%
    filter(!(go_term %in% compartments)) %>%
    filter(pvalue_corrected < 0.05 & direction == "up") %>%
    arrange(pvalue_corrected) %>%
    slice(1:15) %>%
    mutate(go_term = str_wrap(go_term, width = 35)) %>%
    ggplot(aes(x = fct_reorder(go_term, -log10(pvalue_corrected)), y = -log10(pvalue_corrected))) +
        geom_point() +
        ylim(20, 90) +
        labs(x = "Biological Process", y = "-log10(P Value)") +
        # scale_color_gradient(low = "#C5E0B4", high = "#385723") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              plot.title = element_text(size = 15),
              axis.text.y = element_text(size = 12)) +
        coord_flip()
pseudoobscura_mutant_down <- pseudoobscura_mutant_go %>%
    filter(!(go_term %in% compartments)) %>%
    filter(pvalue_corrected < 0.05 & direction == "down") %>%
    arrange(pvalue_corrected) %>%
    slice(1:15) %>%
    mutate(go_term = str_wrap(go_term, width = 35)) %>%
    ggplot(aes(x = fct_reorder(go_term, -log10(pvalue_corrected)), y = -log10(pvalue_corrected))) +
        geom_point() +
        ylim(20, 90) +
        labs(x = "Biological Process", y = "-log10(P Value)") +
        # scale_color_gradient(low = "#C5E0B4", high = "#385723") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              plot.title = element_text(size = 15),
              axis.text.y = element_text(size = 12)) +
        coord_flip()

## add annotation to plots
figure5a <- pseudoobscura_mutant_up + 
    annotate("text", x = 14.5, y = 20, hjust = 0, label = "A", size = 15)
figure5b <- pseudoobscura_mutant_down +
    annotate("text", x = 14.5, y = 20, hjust = 0, label = "B", size = 15)

## plot together
figure5 <- figure5a / figure5b + plot_layout(heights = c(50, 45))
figure5

In [ ]:
ggsave("figures/figure5.png", figure5, height = 13, width = 12)

### Figure 6

In [ ]:
## plot enriched GO terms for adaptive and credible genes
adaptive_dot <- adaptive_go %>%
    filter(evolution == "constraint_adaptive" & pvalue_corrected < 0.05) %>%
    arrange(pvalue_corrected) %>%
    slice(1:15) %>%
    mutate(go_term = str_wrap(go_term, width = 35)) %>%
    ggplot(aes(x = fct_reorder(go_term, -log10(pvalue_corrected)), y = -log10(pvalue_corrected))) +
        geom_point() +
        ylim(0, 35) +
        labs(x = "Biological Process", y = "-log10(P Value)") +
        # scale_color_gradient(low = "#C5E0B4", high = "#385723") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              plot.title = element_text(size = 15),
              axis.text.y = element_text(size = 12)) +
        coord_flip()

credible_dot <- credible_go %>%
    filter(pvalue_corrected < 0.05) %>%
    arrange(pvalue_corrected) %>%
    slice(1:15) %>%
    mutate(go_term = str_wrap(go_term, width = 35)) %>%
    ggplot(aes(x = fct_reorder(go_term, -log10(pvalue_corrected)), y = -log10(pvalue_corrected))) +
        geom_point() +
        ylim(0, 35) +
        labs(x = "Biological Process", y = "-log10(P Value)") +
        # scale_color_gradient(low = "#C5E0B4", high = "#385723") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              plot.title = element_text(size = 15),
              axis.text.y = element_text(size = 12)) +
        coord_flip()

## annotate dot plots for parasperm genes
figure6a <- credible_dot + 
    annotate("text", x = 14.5, y = 32, hjust = 0, label = "A", size = 15)
figure6b <- adaptive_dot +
    annotate("text", x = 13, y = 32, hjust = 0, label = "B", size = 15)

parasperm_dot <- credible_annotated / adaptive_annotated
parasperm_dot

figure6 <- figure6a / figure6b
figure6

In [ ]:
ggsave("figures/figure6.png", figure6, height = 13, width = 12)

### Figure 7

In [ ]:
## now look at credible, separating increase and decrease
credible_up <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
credible_up$p.value

credible_down <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
credible_down$p.value

In [ ]:
## now look at adaptive, separating increase and decrease
adaptive_up <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(adaptive, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
adaptive_up$p.value

adaptive_down <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(adaptive, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
adaptive_down$p.value

In [ ]:
## create graphs comparing deg and phylogenetic parasperm genes
logfc_credible <- annotated_logfc %>%
    filter(credible == "credible") %>%
    na.omit %>%
    ggplot(aes(x = logfc, y = -log10(pvalue), color = direction)) +
        geom_point() +
        labs(title = "Parasperm Branch Associated", x = "LogFC", y = "-log10(P Value)") +
        xlim(-15, 15) +
        ylim(0, 15) +
        scale_color_manual(values = c("steelblue2", "steelblue4", "light gray")) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              plot.title = element_text(hjust = 0.5, size = 18))
logfc_adaptive <- annotated_logfc %>%
    filter(adaptive == "adaptive") %>%
    na.omit %>%
    ggplot(aes(x = logfc, y = -log10(pvalue), color = direction)) +
        geom_point() +
        labs(title = "Parasperm Adaptive", x = "LogFC", y = "-log10(P Value)") +
        xlim(-15, 15) +
        ylim(0, 15) +
        scale_color_manual(values = c("steelblue2", "steelblue4", "light gray")) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              plot.title = element_text(hjust = 0.5, size = 18))

## add graph details to credible
credible_numbers <- annotated_logfc %>%
    filter(credible == "credible") %>%
    count(direction) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    as.list()
up_crediblelabel <- paste0("Up in Mutant = ", credible_numbers["hypo"], "/", length(credible_genes))
up_crediblep <- paste0("p = ", signif(credible_up$p.value, digits = 4))
down_crediblelabel <- paste0("Down in Mutant = ", credible_numbers["hyper"], "/", length(credible_genes))
down_crediblep <- paste0("p = ", signif(credible_down$p.value, digits = 4))

figure7a <- logfc_credible +
    annotate("text", x = -15, y = 14, hjust = 0,
             size = 10, label = "A") +
    annotate("text", x = -15, y = 12, hjust = 0,
             size = 2, label = up_crediblelabel) +
    annotate("text", x = -14, y = 11.5, hjust = 0,
             size = 2, label = up_crediblep) +
    annotate("text", x = -15, y = 10, hjust = 0,
             size = 2, label = down_crediblelabel) +
    annotate("text", x = -14, y = 9.5, hjust = 0,
             size = 2, label = down_crediblep)

## add graph details to adaptive
adaptive_numbers <- annotated_logfc %>%
    filter(adaptive == "adaptive") %>%
    count(direction) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    as.list()
up_adaptivelabel <- paste0("Up in Mutant = ", adaptive_numbers["hyper"], "/", length(adaptive_genes))
up_adaptivep <- paste0("p = ", signif(adaptive_up$p.value, digits = 4))
down_adaptivelabel <- paste0("Down in Mutant = ", adaptive_numbers["hypo"], "/", length(adaptive_genes))
down_adaptivep <- paste0("p = ", signif(adaptive_down$p.value, digits = 4))

figure7b <- logfc_adaptive +
    annotate("text", x = -15, y = 14, hjust = 0,
             size = 10, label = "B") +
    annotate("text", x = -15, y = 12, hjust = 0,
             size = 2, label = up_adaptivelabel) +
    annotate("text", x = -14, y = 11.5, hjust = 0,
             size = 2, label = up_adaptivep) +
    annotate("text", x = -15, y = 10, hjust = 0,
             size = 2, label = down_adaptivelabel) +
    annotate("text", x = -14, y = 9.5, hjust = 0,
             size = 2, label = down_adaptivep)

## plot together
figure7ab <- figure7a + figure7b
figure7ab

In [ ]:
ggsave("figures/figure7ab.png", figure7ab,
       height = 4, width = 8)

In [ ]:
upcred_dot <- upcred_go %>%
    filter(pvalue_corrected < 0.05) %>%
    arrange(pvalue_corrected) %>%
    slice(1:15) %>%
    mutate(go_term = str_wrap(go_term, width = 35)) %>%
    ggplot(aes(x = fct_reorder(go_term, -log10(pvalue_corrected)), y = -log10(pvalue_corrected))) +
        geom_point() +
        ylim(0, 75) +
        labs(x = "Biological Process", y = "-log10(P Value)") +
        # scale_color_gradient(low = "#C5E0B4", high = "#385723") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              plot.title = element_text(size = 15),
              axis.text.y = element_text(size = 12)) +
        coord_flip()
figure7c <- upcred_dot +
    annotate("text", x = 15, y = 0, hjust = 0, label = "C", size = 10)

figure7c

In [ ]:
ggsave("figures/figure7c.png", figure7c, height = 6, width = 8)

## Create Supplemental Figures

### Supplemental 1

In [ ]:
## plot clustering information
supplemental1 <- k_df %>%
    ggplot(aes(x = wcss, y = avg_number, color = stdev)) +
    geom_point() +
    labs(title = "Optimization of WCSS and Average Number of Genes in Each Cluster for Each K",
         x = "Within Cluster Sum of Squares", y = "Average Number of Genes in Each Cluster",
         color = "Standard Deviation\nof Number of Genes\nin Each Cluster") +
    theme_bw() +
    theme(panel.grid = element_blank())
supplemental1

In [ ]:
## save the k statistic plot
ggsave("figures/supplemental1.png", supplemental1, width = 10, height = 6)

### Supplemental 2

In [ ]:
## get the number of mutations per gene
regulator <- c("upstream_gene_variant", "downstream_gene_variant", "3_prime_UTR_variant", "5_prime_UTR_variant")

multi_mutations <- variants %>%
    separate_longer_delim(consequence, ",") %>%
    filter(biotype == "protein_coding" & consequence != "synonymous_variant") %>%
    mutate(mutation_type = case_when(consequence %in% regulator ~ "Regulatory Element",
                             consequence == "intron_variant" ~ "Intron Variant",
                             grepl("splice", consequence) ~ "Splicing Variant",
                             grepl("inframe", consequence) ~ "Inframe InDel",
                             grepl("frameshift", consequence) ~ "Frameshift InDel",
                             consequence == "protein_altering_variant" ~ "Protein Altering",
                             consequence == "stop_gained" ~ "Nonsense Mutation",
                             consequence == "start_lost" ~ "Start Lost",
                             TRUE ~ "Other")) %>%
    group_by(mutation_type, symbol) %>%
    reframe(count = n())

## find limit that 99% of genes are within
cap <- quantile(multi_mutations$count, probs = 0.75)
attributes(cap) <- NULL

multi_mutations_graph <- multi_mutations %>%
    ggplot(aes(x = count)) +
        geom_density() +
        labs(title = "A",
             x = "Number of Mutations", y = "Density of Genes") +
        xlim(1, cap) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              plot.title = element_text(size = 20))
multi_mutations_type_graph <- multi_mutations %>%
    ggplot(aes(x = count)) +
        geom_density(alpha = 0.5) +
        labs(title = "B",
             x = "Number of Mutations", y = "Density of Genes") +
        # scale_color_viridis(discrete = TRUE) +
        # scale_fill_viridis(discrete = TRUE) +
        xlim(1, cap) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              plot.title = element_text(size = 20)) + 
        facet_wrap(vars(mutation_type), scales = "free")
supplemental2 <- plot_grid(multi_mutations_graph, multi_mutations_type_graph, ncol = 1)
supplemental2

In [ ]:
ggsave("figures/supplemental2.png", supplemental2, height = 15, width = 10)

### Supplemental 3

In [ ]:
## add tau information to comparison data
tau_comparison <- tau %>%
    full_join(comparison_norm, by = "gene") %>%
    mutate(tau_tissue = str_replace_all(tau_tissue, "_", " "),
           direction = case_when(significance == "significant" & logfc > 0 ~ "Increased",
                                 significance == "significant" & logfc < 0 ~ "Decreased",
                                 TRUE ~ "Not Significant"),
           comparison = case_when(comparison == "reference_mutant" ~ "Wildtype vs. Mutant",
                                  comparison == "sex ratio_reference" ~ "Sex Ratio vs. Reference",
                                  comparison == "sex ratio_mutant" ~ "Sex Ratio vs. Mutant")) %>%
    na.omit()

## plot the distribution of tau based on the specific tissue and the comparison
supplemental3a <- tau_comparison %>%
    ggplot(aes(x = tau, fill = tau_tissue)) +
        geom_histogram() +
        scale_fill_viridis(discrete = TRUE) +
        labs(title = "A",
             x = "Tissue Specificity", y = "Number of Genes", fill = "Assigned Tissue") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              text = element_text(size = 10),
              plot.title = element_text(size = 22))
supplemental3b <- tau_comparison %>%
    ggplot(aes(x = tau, fill = tau_tissue)) +
        geom_histogram() +
        scale_fill_viridis(discrete = TRUE) +
        labs(title = "B",
             x = "Tissue Specificity", y = "Number of Genes", fill = "Assigned Tissue") +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "none",
              text = element_text(size = 10),
              plot.title = element_text(size = 22)) +
        facet_grid(cols = vars(direction),
                   rows = vars(comparison))

## plot the two graphs together
supplemental3 <- supplemental3a / supplemental3b
supplemental3

In [ ]:
ggsave("figures/supplemental3.png", supplemental3, width = 10, height = 12, units = "in")

### Supplemental 4

In [ ]:
## pull adaptive and constrained genes
adaptive_genes <- evogenex %>%
    filter(constraint == "constraint" & adaptive == "adaptive") %>%
    pull("gene")
constrained_genes <- evogenex %>%
    filter(constraint == "constraint") %>%
    pull("gene")
nonparasperm_genes <- reads$gene_id[!(reads$gene_id %in% adaptive_genes | reads$gene_id %in% credible_genes)]

In [ ]:
## create list of genes that are adaptive or credible
venn_list <- list(
    credible = credible_genes,
    adaptive = adaptive_genes
)

## create venn diagram of genes that are adaptive and/or credible
parasperm_venn <- ggvenn(
    venn_list,
    # fill_color = c(),
    stroke_size = 0.5,
    set_name_size = 4
)

supplemental4a <- parasperm_venn +
    labs(title = "A") +
    theme(plot.title = element_text(size = 22))

In [ ]:
## create list of genes and their assigned cluster
cluster_count <- pic_clusters %>%
    count(cluster, sort = TRUE)

## get list of clusters with genes of interest
credible_clusters <- pic_clusters %>%
    filter(gene %in% credible_genes) %>%
    pull("cluster") %>%
    unique()
constrained_clusters <- pic_clusters %>%
    filter(gene %in% constrained_genes) %>%
    pull("cluster") %>%
    unique()
adaptive_clusters <- _picclusters %>%
    filter(gene %in% adaptive_genes) %>%
    pull("cluster") %>%
    unique()

## annotate clusters with attributes, if has one assigned gene
cluster_annotated <- cluster_count %>%
    rename(count = n) %>%
    mutate(credible = case_when(cluster %in% credible_clusters ~ "credible",
                                TRUE ~ "non-credible"),
           constrained = case_when(cluster %in% constrained_clusters ~ "constrained",
                                   TRUE ~ "neutral"),
           adaptive = case_when(cluster %in% adaptive_clusters ~ "adaptive",
                                TRUE ~ "non-adaptive"))

## add grouped annotated information
clusters_annotated <- pic_clusters %>%
    filter(n > 5) %>%
    mutate(cluster = as.integer(cluster)) %>%
    left_join(cluster_annotated, by = "cluster") %>%
    mutate(attr = paste(credible, adaptive, sep = ", "))

## plot the ones with different attributes
supplemental4b <- clusters_annotated %>%
    filter(attr != "non-credible, non-adaptive") %>%
    mutate(attr = str_replace(attr, "_", ", "),
           attr = str_replace(attr, "_", ", ")) %>%
    ggplot(aes(x = parasperm, y = scale_change, color = attr, group = gene)) +
        geom_line() +
        labs(title = "B", 
             x = "Parasperm Percentage", y = "Scaled Gene Expression", color = "Parasperm Association") +
        scale_color_manual(values = c("green", "blue", "yellow")) + 
        facet_wrap(vars(name)) +
        theme_bw() +
        theme(panel.grid = element_blank(),
              legend.position = "bottom",
              text = element_text(size = 8),
              axis.title = element_text(size = 15),
              plot.title = element_text(size = 22))

In [ ]:
supplemental4 <- supplemental4a / supplemental4b
supplemental4

In [ ]:
ggsave("figures/supplemental4.png", supplemental4, height = 15, width = 20)